# 01 — Build the dataset (reference year 2023)
Sources: UNDP HDI 2023 (`hdr.xlsx`) and World Bank WDI 2023 (`WDI_2023.csv`, `WDI_country_metadata.csv`). Only 2023 values are used (no look-back).

In [1]:
import pandas as pd

## HDI

In [2]:
hdr = pd.read_excel("../data/raw/hdr.xlsx")
hdi = hdr[hdr["indicatorCode"] == "hdi"]
hdi = hdi[["countryIsoCode", "value"]]
hdi.columns = ["ISO3", "HDI"]

# HDR aggregates have longer codes such as ZZK.WORLD
hdi = hdi[hdi["ISO3"].str.len() == 3]

# UNDP codes Nauru as NRO; the ISO / World Bank code is NRU
hdi["ISO3"] = hdi["ISO3"].replace("NRO", "NRU")
len(hdi)

193

## WDI indicators and region

In [3]:
codes = {
    "NY.GDP.MKTP.KD.ZG": "GDP_growth",
    "FP.CPI.TOTL.ZG": "Inflation",
    "SL.UEM.TOTL.ZS": "Unemployment",
    "SP.DYN.IMRT.IN": "Infant_mortality",
    "EG.ELC.ACCS.ZS": "Electricity_access",
    "IT.NET.USER.ZS": "Internet_users",
    "NV.AGR.TOTL.ZS": "Agriculture_VA",
    "NV.IND.TOTL.ZS": "Industry_VA",
    "NE.TRD.GNFS.ZS": "Trade",
}

wdi = pd.read_csv("../data/raw/WDI_2023.csv")
wdi = wdi[wdi["Indicator Code"].isin(codes.keys())]
wdi = wdi.pivot(index="Country Code", columns="Indicator Code", values="Value_2023")
wdi = wdi.rename(columns=codes)

meta = pd.read_csv("../data/raw/WDI_country_metadata.csv")
meta = meta[meta["Region"].notna()]          # aggregates have no region
meta = meta[["Country Code", "Table Name", "Region"]]
meta.columns = ["ISO3", "Country", "Region"]
len(meta)

217

## Merge on ISO3

In [4]:
df = meta.merge(wdi, left_on="ISO3", right_index=True, how="left")
df = df.merge(hdi, on="ISO3", how="left")

# countries lost because they have no HDI value
df[df["HDI"].isna()]["Country"]

0                          Aruba
8                 American Samoa
24                       Bermuda
34               Channel Islands
46                       Curaçao
47                Cayman Islands
65                 Faroe Islands
71                     Gibraltar
78                     Greenland
80                          Guam
88                   Isle of Man
118             Macao SAR, China
119     St. Martin (French part)
121                       Monaco
133     Northern Mariana Islands
140                New Caledonia
157             Puerto Rico (US)
158    Korea, Dem. People's Rep.
162             French Polynesia
184    Sint Maarten (Dutch part)
187     Turks and Caicos Islands
207       British Virgin Islands
208          Virgin Islands (US)
212                       Kosovo
Name: Country, dtype: str

In [5]:
df = df[df["HDI"].notna()]

# HDR countries that did not match (should be empty)
hdi[~hdi["ISO3"].isin(df["ISO3"])]

,ISO3,HDI


In [6]:
len(df)

193

## Target
$y = 0$ (Low) if HDI $< 0.550$, $y = 1$ (Medium/High) if $0.550 \le$ HDI $< 0.800$, $y = 2$ (Very High) if HDI $\ge 0.800$.

In [7]:
df["y"] = 1
df.loc[df["HDI"] < 0.550, "y"] = 0
df.loc[df["HDI"] >= 0.800, "y"] = 2
df["HDI_class"] = df["y"].map({0: "Low", 1: "Medium/High", 2: "Very High"})
df["HDI_class"].value_counts()

HDI_class
Medium/High    93
Very High      74
Low            26
Name: count, dtype: int64

## DominantSector
Services share $= 100 - \text{agriculture} - \text{industry}$. If an input is missing or a share is negative, the sector is left missing.

In [8]:
agr = df["Agriculture_VA"]
ind = df["Industry_VA"]
srv = 100 - agr - ind
valid = (agr >= 0) & (ind >= 0) & (srv >= 0)

df["DominantSector"] = None
df.loc[valid & (agr > ind) & (agr > srv), "DominantSector"] = "Agriculture"
df.loc[valid & (ind > agr) & (ind > srv), "DominantSector"] = "Industry"
df.loc[valid & (srv > agr) & (srv > ind), "DominantSector"] = "Services"
df["DominantSector"].value_counts(dropna=False)

DominantSector
Services       173
None            10
Industry         8
Agriculture      2
Name: count, dtype: int64

## Missingness (%)

In [9]:
df.isna().mean() * 100

ISO3                   0.000000
Country                0.000000
Region                 0.000000
Electricity_access     0.000000
Inflation              9.326425
Internet_users         7.253886
Trade                 13.471503
Agriculture_VA         5.181347
Industry_VA            5.181347
GDP_growth             2.590674
Unemployment           8.808290
Infant_mortality       1.036269
HDI                    0.000000
y                      0.000000
HDI_class              0.000000
DominantSector         5.181347
dtype: float64

## Save
`HDI` is kept only to show where the target comes from; it is never used as a predictor. See `data_dictionary.md` for column descriptions.

In [10]:
cols = ["ISO3", "Country", "Region", "GDP_growth", "Inflation", "Unemployment", "Infant_mortality",
        "Electricity_access", "Internet_users", "Agriculture_VA", "Industry_VA", "Trade",
        "DominantSector", "HDI", "y", "HDI_class"]
df = df[cols].sort_values("ISO3")
df.to_csv("../data/processed/hdi_classification_2023.csv", index=False)